# Lightweight Multi-Task Brain MRI CNN
All modules assembled into a single notebook.
**Do not delete the original .py files.**

In [ ]:
# Install dependencies (CPU-only: no CUDA packages needed)
# Added joblib for parallel preprocessing; torch CPU build is sufficient
!pip install -q torch torchvision timm albumentations opencv-python scikit-learn seaborn onnx onnxruntime grad-cam tensorboard joblib


In [9]:
pip install --upgrade timm

Note: you may need to restart the kernel to use updated packages.


## Class balance Data

In [ ]:
# import os
# import shutil
# import random
# import math

# base_dir = r"/kaggle/input/datasets/n00raayy/lightweighttesting/FullDataset"
# new_dir = r"/kaggle/input/datasets/n00raayy"
# output_dir = "/kaggle/working/Ready-Dataset"

# # Define sources, target destinations, and their desired fractions of the total dataset
# # Note: real classes use 'no_tumor' in the workspace but we can rename it to 'notumor' in the target as requested
# sources = [
#     # 1. real Data (80% total -> 20% each)
#     ("real/glioma", "real/glioma", 0.20),
#     ("real/no_tumor", "real/notumor", 0.20),
#     ("real/meningioma", "real/meningioma", 0.20),
#     ("real/pituitary", "real/pituitary", 0.20),
    
#     # 2. fake Data (20%) 
#     # 2.1 gan-generated data (10% total -> 2.5% each)
#     ("fake/gan/glioma", "fake/gan/glioma", 0.025),
#     ("fake/gan/notumor", "fake/gan/notumor", 0.025),
#     ("fake/gan/meningioma", "fake/gan/meningioma", 0.025),
#     ("fake/gan/pituitary", "fake/gan/pituitary", 0.025),
    
#     # 2.2 attacks Attacks (10% total -> 5% PGD, 5% FGSM)
#     # PGD (5% total -> 1.25% each)
#     ("fake/attacks/pgd/glioma", "fake/attacks/pgd/glioma", 0.0125),
#     ("fake/attacks/pgd/notumor", "fake/attacks/pgd/notumor", 0.0125),
#     ("fake/attacks/pgd/meningioma", "fake/attacks/pgd/meningioma", 0.0125),
#     ("fake/attacks/pgd/pituitary", "fake/attacks/pgd/pituitary", 0.0125),
    
#     # FGSM (5% total -> 1.25% each)
#     ("fake/attacks/fgsm/glioma", "fake/attacks/fgsm/glioma", 0.0125),
#     ("fake/attacks/fgsm/notumor", "fake/attacks/fgsm/notumor", 0.0125),
#     ("fake/attacks/fgsm/meningioma", "fake/attacks/fgsm/meningioma", 0.0125),
#     ("fake/attacks/fgsm/pituitary", "fake/attacks/fgsm/pituitary", 0.0125),
# ]

# # Calculate the maximum possible dataset size based on the most limiting folder
# max_n = float('inf')
# counts_info = {}
# limiting_class = ""

# for src_rel, dest_rel, fraction in sources:
#     src_path = os.path.join(base_dir, os.path.normpath(src_rel))
#     if os.path.exists(src_path):
#         # Count files
#         files = [f for f in os.listdir(src_path) if os.path.isfile(os.path.join(src_path, f))]
#         count = len(files)
#         counts_info[src_rel] = {'count': count, 'files': files}
        
#         # Calculate maximum possible total dataset size if this directory is the bottleneck
#         possible_total = count / fraction
#         if possible_total < max_n:
#             max_n = possible_total
#             limiting_class = src_rel
#     else:
#         print(f"Warning: source path {src_path} not found.")

# max_n = math.floor(max_n)
# print(f"Limiting class is '{limiting_class}' which allows a maximum total dataset size of {max_n} images.\n")

# if max_n > 0:
#     total_images_to_extract = max_n
    
#     print("Started copying and balancing dataset:")
#     # Process copying
#     for src_rel, dest_rel, fraction in sources:
#         target_count = int(total_images_to_extract * fraction)
#         files = counts_info[src_rel]['files']
        
#         # Sample randomly
#         target_count = min(target_count, len(files))
#         sampled_files = random.sample(files, target_count)
        
#         dest_path = os.path.join(output_dir, os.path.normpath(dest_rel))
#         os.makedirs(dest_path, exist_ok=True)
        
#         for file in sampled_files:
#             src_file = os.path.join(base_dir, os.path.normpath(src_rel), file)
#             dest_file = os.path.join(dest_path, file)
#             shutil.copy(src_file, dest_file)
            
#         print(f"  -> Copied {target_count} images to {dest_rel} ({fraction*100}%)")
        
#     print(f"\n✅ Ready-Dataset compilation successfully finished at: {output_dir}")
# else:
#     print("Error: Could not determine total images due to missing files or empty directories.")

In [11]:
!ls /kaggle/working/

checkpoints  export  Ready-Dataset  results  runs


In [1]:
print("Setup complete. Ready to run the training notebook.")

Setup complete. Ready to run the training notebook.


---
## config.py

In [2]:
"""
config.py
---------
Central configuration for the Lightweight Multi-Task Brain MRI CNN.
All hyper-parameters, paths and class definitions are defined here.

CPU-PARALLELISM CHANGES:
  - device default changed from "cuda" to "cpu"
  - amp (automatic mixed precision) disabled (not supported on CPU)
  - pin_memory disabled (only useful with CUDA)
  - num_workers set to os.cpu_count() for optimal parallel data loading
"""

import os
from dataclasses import dataclass, field
from typing import List, Optional


# ─────────────────────────────────────────────────────────────────────────────
# Class Definitions
# ─────────────────────────────────────────────────────────────────────────────

# Authenticity labels  (head 1)
AUTHENTICITY_CLASSES = ["real", "fake"]          # binary
AUTHENTICITY_IDX     = {c: i for i, c in enumerate(AUTHENTICITY_CLASSES)}

# Tumor-type labels   (head 2, only for real images)
TUMOR_CLASSES  = ["no_tumor", "glioma", "meningioma", "pituitary"]
TUMOR_IDX      = {c: i for i, c in enumerate(TUMOR_CLASSES)}
NUM_TUMOR_CLASSES = len(TUMOR_CLASSES)


# ─────────────────────────────────────────────────────────────────────────────
# 5-way flat label space used for the joint confusion matrix
# ─────────────────────────────────────────────────────────────────────────────
JOINT_CLASSES = ["no_tumor", "glioma", "meningioma", "pituitary", "fake"]
NUM_JOINT_CLASSES = len(JOINT_CLASSES)  # 5


@dataclass
class DataConfig:
    """Dataset and augmentation settings."""
    data_root: str        = "../brisc-deepfakes/ReadyDataset"
    image_size: int       = 224
    mean: List[float]     = field(default_factory=lambda: [0.485, 0.456, 0.406])
    std:  List[float]     = field(default_factory=lambda: [0.229, 0.224, 0.225])

    aug_rotation_limit:  int   = 20
    aug_brightness:      float = 0.2
    aug_contrast:        float = 0.2
    aug_noise_var_limit: tuple = (10.0, 50.0)
    aug_elastic_alpha:   int   = 120
    aug_elastic_sigma:   int   = 6

    val_split:  float = 0.10
    test_split: float = 0.20
    k_folds:    int   = 5
    random_seed: int  = 42


@dataclass
class ModelConfig:
    """Architecture settings."""
    backbone: str = "lcnet_050"
    pretrained: bool = True
    freeze_backbone_epochs: int = 5
    dropout_rate: float     = 0.5
    embedding_dim: int      = 512

    auth_head_hidden:      int = 256
    tumor_head_hidden:     int = 256


@dataclass
class TrainingConfig:
    """Training and optimiser settings."""
    stage1_epochs: int   = 20
    stage1_lr:     float = 1e-3
    stage1_batch:  int   = 32

    stage2_epochs: int   = 15
    stage2_lr:     float = 5e-4
    stage2_batch:  int   = 32

    stage3_epochs: int   = 30
    stage3_lr:     float = 1e-4
    stage3_batch:  int   = 16

    loss_weight_auth:      float = 1.0
    loss_weight_tumor:     float = 1.5

    optimizer:     str   = "adam"
    weight_decay:  float = 1e-3
    scheduler:     str   = "cosine"
    warmup_epochs: int   = 3

    # ── CPU parallelism: use all logical cores for data loading ───────────────
    # os.cpu_count() returns 8 on a 4-core/8-thread i7 8th Gen.
    # Using all threads saturates I/O and preprocessing without over-subscribing.
    num_workers:   int   = os.cpu_count() or 4   # changed: was hard-coded 4

    # pin_memory only benefits CUDA transfers; disable on CPU to save RAM
    pin_memory:    bool  = False                  # changed: was True

    # AMP (fp16) is not supported on CPU; keep False to avoid runtime errors
    mixed_precision: bool = False                 # changed: was True

    grad_clip:     float = 1.0
    early_stop_patience: int = 10


@dataclass
class EvalConfig:
    """Evaluation and export settings."""
    metrics: List[str] = field(default_factory=lambda: [
        "accuracy", "f1_macro", "f1_per_class", "auc_roc", "confusion_matrix"
    ])
    grad_cam_layer: str = "backbone.conv5"

    export_onnx:        bool = True
    onnx_opset:         int  = 17
    quantize_int8:      bool = True


@dataclass
class PathConfig:
    """Output path settings."""
    checkpoint_dir: str = "checkpoints"
    log_dir:        str = "runs"
    onnx_dir:       str = "export"
    results_dir:    str = "results"

    def make_dirs(self):
        for d in [self.checkpoint_dir, self.log_dir,
                  self.onnx_dir, self.results_dir]:
            os.makedirs(d, exist_ok=True)


@dataclass
class Config:
    """Master config – compose all sub-configs."""
    data:     DataConfig     = field(default_factory=DataConfig)
    model:    ModelConfig    = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    eval:     EvalConfig     = field(default_factory=EvalConfig)
    paths:    PathConfig     = field(default_factory=PathConfig)
    device:   str            = "cpu"   # changed: was "cuda" — no GPU on this machine
    amp:      bool           = False   # changed: was True — AMP not supported on CPU
    seed:     int            = 42


# Singleton used across the project
CFG = Config()

# ── Report effective parallelism settings ─────────────────────────────────────
import multiprocessing
print(f"[Config] CPU cores available : {multiprocessing.cpu_count()}")
print(f"[Config] DataLoader workers  : {CFG.training.num_workers}")
print(f"[Config] Device              : {CFG.device}")
print(f"[Config] Mixed precision     : {CFG.amp}")


[Config] CPU cores available : 8
[Config] DataLoader workers  : 8
[Config] Device              : cpu
[Config] Mixed precision     : False


---
## utils.py

In [3]:
"""
utils.py
--------
Shared utilities:
  - set_seed
  - count_parameters
  - EarlyStopping class
  - LabelSmoothingCrossEntropy
  - Mixup augmentation
  - AverageMeter
  - format_time

CPU-PARALLELISM CHANGES:
  - set_seed: removed cuda-specific calls (safe to call even without GPU)
  - WarmupScheduler: no change needed
"""

from __future__ import annotations

import math
import random
import time
from typing import Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ─────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # torch.cuda.manual_seed_all() skipped — no GPU; safe to omit on CPU
    # deterministic mode still set for reproducibility across CPU threads
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────────────────────────────────────
# Parameter counting
# ─────────────────────────────────────────────────────────────────────────────

def count_parameters(model: nn.Module) -> Tuple[int, int]:
    """Returns (total_params, trainable_params)."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# ─────────────────────────────────────────────────────────────────────────────
# Early Stopping
# ─────────────────────────────────────────────────────────────────────────────

class EarlyStopping:
    """
    Stop training when monitored metric stops improving.

    Parameters
    ----------
    patience  : epochs to wait after last improvement
    min_delta : minimum change to qualify as improvement
    mode      : 'min' (loss) or 'max' (accuracy)
    """

    def __init__(self, patience: int = 10, min_delta: float = 1e-4,
                 mode: str = "min"):
        self.patience  = patience
        self.min_delta = min_delta
        self.mode      = mode
        self.best      = float("inf") if mode == "min" else -float("inf")
        self.counter   = 0
        self.triggered = False

    def __call__(self, metric: float) -> bool:
        if self.mode == "min":
            improved = metric < self.best - self.min_delta
        else:
            improved = metric > self.best + self.min_delta

        if improved:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


# ─────────────────────────────────────────────────────────────────────────────
# Label Smoothing Cross-Entropy
# ─────────────────────────────────────────────────────────────────────────────

class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy with label smoothing. Handles ignore_index=-1.

    Parameters
    ----------
    smoothing    : float in [0, 1), 0 = standard CE
    ignore_index : label to ignore (e.g. masked samples)
    """

    def __init__(self, smoothing: float = 0.1, ignore_index: int = -1):
        super().__init__()
        self.smoothing     = smoothing
        self.ignore_index  = ignore_index

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        mask = (targets != self.ignore_index)
        if not mask.any():
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        logits  = logits[mask]
        targets = targets[mask]

        n_classes = logits.size(-1)
        log_probs = F.log_softmax(logits, dim=-1)

        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs,
                                             self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing)

        return -(smooth_targets * log_probs).sum(dim=-1).mean()


# ─────────────────────────────────────────────────────────────────────────────
# Mixup augmentation
# ─────────────────────────────────────────────────────────────────────────────

def mixup_data(x: torch.Tensor, y: torch.Tensor,
               alpha: float = 0.2) -> Tuple[torch.Tensor, torch.Tensor,
                                            torch.Tensor, float]:
    """Apply Mixup to a batch (x, y). Returns (mixed_x, y_a, y_b, lam)."""
    if alpha > 0:
        lam = float(np.random.beta(alpha, alpha))
    else:
        lam = 1.0

    batch_size = x.size(0)
    idx        = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute Mixup loss."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ─────────────────────────────────────────────────────────────────────────────
# AverageMeter
# ─────────────────────────────────────────────────────────────────────────────

class AverageMeter:
    """Tracks running average of a scalar metric."""

    def __init__(self, name: str = ""):
        self.name = name
        self.reset()

    def reset(self):
        self.val   = 0.0
        self.avg   = 0.0
        self.sum   = 0.0
        self.count = 0

    def update(self, val: float, n: int = 1):
        self.val    = val
        self.sum   += val * n
        self.count += n
        self.avg    = self.sum / self.count

    def __str__(self):
        return f"{self.name}: {self.avg:.4f}"


# ─────────────────────────────────────────────────────────────────────────────
# Time formatting
# ─────────────────────────────────────────────────────────────────────────────

def format_time(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


# ─────────────────────────────────────────────────────────────────────────────
# Warmup LR scheduler wrapper
# ─────────────────────────────────────────────────────────────────────────────

class WarmupScheduler:
    """
    Linear warmup for the first `warmup_steps` steps,
    then delegates to `base_scheduler`.
    """

    def __init__(self, optimizer, warmup_steps: int, base_scheduler):
        self.optimizer      = optimizer
        self.warmup_steps   = warmup_steps
        self.base_scheduler = base_scheduler
        self._step          = 0
        self._base_lrs      = [pg["lr"] for pg in optimizer.param_groups]
        for pg in optimizer.param_groups:
            pg["lr"] = 1e-8

    def step(self):
        self._step += 1
        if self._step <= self.warmup_steps:
            frac = self._step / self.warmup_steps
            for pg, base_lr in zip(self.optimizer.param_groups, self._base_lrs):
                pg["lr"] = base_lr * frac
        else:
            self.base_scheduler.step()


---
## dataset.py

In [4]:
"""
dataset.py
----------
MRI multi-task dataset with:
  - MRI-specific augmentation (intensity normalization, noise, elastic)
  - Balanced sampling via WeightedRandomSampler
  - Patient-level train/val/test splitting (no data leakage)
  - Returns (image, auth_label, tumor_label, joint_label)
    where tumor_label = -1 for fake images (masked in loss)

CPU-PARALLELISM CHANGES:
  1. scan_dataset_parallel: uses concurrent.futures.ThreadPoolExecutor to
     scan sub-directories in parallel (I/O bound — threads beat processes here).
  2. parallel_preload_cache: optional joblib Parallel preprocessing to warm the
     OS file cache before training begins, using all available CPU cores.
  3. DataLoader num_workers comes from CFG (set to os.cpu_count() in config.py),
     enabling multi-process data loading with one worker per logical thread.
  4. prefetch_factor=2 added to DataLoader: each worker pre-fetches 2 batches
     ahead of the training loop, hiding disk latency on CPU.
  5. persistent_workers=True: worker processes are reused across epochs,
     avoiding fork/join overhead each epoch.
"""

import os
import random
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed  # NEW: parallel I/O

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from joblib import Parallel, delayed  # NEW: parallel CPU preprocessing


# ─────────────────────────────────────────────────────────────────────────────
# Augmentation pipelines
# ─────────────────────────────────────────────────────────────────────────────

def build_train_transforms(cfg=CFG) -> A.Compose:
    """Heavy MRI-specific augmentation for training."""
    dc = cfg.data
    return A.Compose([
        A.Resize(dc.image_size, dc.image_size),

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.Rotate(limit=dc.aug_rotation_limit, p=0.7,
                 border_mode=cv2.BORDER_REFLECT_101),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=0, p=0.4),
        A.ElasticTransform(alpha=dc.aug_elastic_alpha,
                           sigma=dc.aug_elastic_sigma,
                           p=0.3),

        A.RandomBrightnessContrast(brightness_limit=dc.aug_brightness,
                                   contrast_limit=dc.aug_contrast, p=0.6),
        A.GaussNoise(var_limit=dc.aug_noise_var_limit, p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),

        A.CoarseDropout(max_holes=8, max_height=16, max_width=16,
                        fill_value=0, p=0.2),

        A.Normalize(mean=dc.mean, std=dc.std),
        ToTensorV2(),
    ])


def build_val_transforms(cfg=CFG) -> A.Compose:
    """Deterministic pipeline for validation / test."""
    dc = cfg.data
    return A.Compose([
        A.Resize(dc.image_size, dc.image_size),
        A.Normalize(mean=dc.mean, std=dc.std),
        ToTensorV2(),
    ])


# ─────────────────────────────────────────────────────────────────────────────
# Sample catalogue builder — CPU-PARALLEL VERSION
# ─────────────────────────────────────────────────────────────────────────────

def _joint_label(auth: int, tumor: int) -> int:
    if auth == AUTHENTICITY_IDX["real"]:
        return tumor  # 0..3
    else:
        return 4  # fake (single class)


def _scan_subdir(subdir: Path) -> Optional[Dict]:
    """
    Process a single file path into a sample dict.
    Designed to be called in parallel via ThreadPoolExecutor.

    Threads are used (not processes) because this is I/O-bound directory
    traversal; threads share memory safely and avoid pickling overhead.
    """
    if not subdir.is_file():
        return None
    if subdir.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
        return None

    parts = [p.lower() for p in subdir.parts]

    if "real" in parts:
        auth = AUTHENTICITY_IDX["real"]
        folder = subdir.parent.name.lower()
        tumor_map = {
            "notumor":   TUMOR_IDX["no_tumor"],
            "no_tumor":  TUMOR_IDX["no_tumor"],
            "glioma":    TUMOR_IDX["glioma"],
            "meningioma":TUMOR_IDX["meningioma"],
            "pituitary": TUMOR_IDX["pituitary"],
        }
        tumor_label = tumor_map.get(folder, -1)
        if tumor_label == -1:
            return None
    elif "fake" in parts:
        auth = AUTHENTICITY_IDX["fake"]
        tumor_label = -1
    else:
        return None

    joint = _joint_label(auth, tumor_label)
    return {"path": str(subdir), "auth": auth, "tumor": tumor_label, "joint": joint}


def scan_dataset(root: str, max_workers: int = None) -> List[Dict]:
    """
    Walk the directory tree and build a flat list of sample dicts.

    CPU-PARALLEL: Uses ThreadPoolExecutor to scan files concurrently.
    max_workers defaults to os.cpu_count() (8 on i7-8th-Gen).

    Expected layout::
        <root>/
          real/  notumor/ glioma/ meningioma/ pituitary/
          fake/  gan/     attacks/
    """
    if max_workers is None:
        max_workers = os.cpu_count() or 4

    root_path = Path(root)
    # Collect all candidate file paths first (fast rglob, single-threaded)
    all_paths = list(root_path.rglob("*"))

    samples: List[Dict] = []
    # Dispatch file-classification to a thread pool (I/O + stat calls)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_scan_subdir, p): p for p in all_paths}
        for future in as_completed(futures):
            result = future.result()
            if result is not None:
                samples.append(result)

    return samples


# ─────────────────────────────────────────────────────────────────────────────
# Optional: parallel image pre-load to warm OS cache before training
# ─────────────────────────────────────────────────────────────────────────────

def _load_one(path: str, image_size: int) -> None:
    """
    Read, decode, and resize one image — result discarded.
    Side-effect: file is now in the OS page cache, so subsequent
    DataLoader workers read from RAM instead of disk.
    """
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is not None:
        cv2.resize(img, (image_size, image_size))


def parallel_preload_cache(samples: List[Dict],
                            image_size: int = 224,
                            n_jobs: int = -1) -> None:
    """
    Warm the OS file cache by reading every image in parallel using joblib.

    Parameters
    ----------
    samples    : list of sample dicts (must have "path" key)
    image_size : resize target (matches training size)
    n_jobs     : joblib parallel workers; -1 = use all CPU cores
                 On i7-8th-Gen this spawns up to 8 parallel readers.

    Usage (optional, call before build_dataloaders for faster first epoch):
        parallel_preload_cache(all_samples, image_size=CFG.data.image_size)
    """
    paths = [s["path"] for s in samples]
    print(f"[PreloadCache] Warming OS cache for {len(paths):,} images "
          f"(n_jobs={n_jobs}) …")
    # loky backend (default) spawns separate processes — safe for cv2
    Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_load_one)(p, image_size) for p in paths
    )
    print("[PreloadCache] Done.")


# ─────────────────────────────────────────────────────────────────────────────
# Patient-level stratified split
# ─────────────────────────────────────────────────────────────────────────────

def stratified_patient_split(
    samples: List[Dict],
    val_frac: float = 0.15,
    test_frac: float = 0.10,
    seed: int = 42,
) -> Tuple[List[Dict], List[Dict], List[Dict]]:
    """Split samples into train / val / test."""
    rng = random.Random(seed)

    from collections import defaultdict
    buckets: Dict[int, List[Dict]] = defaultdict(list)
    for s in samples:
        buckets[s["joint"]].append(s)

    train_list, val_list, test_list = [], [], []

    for label, group in buckets.items():
        rng.shuffle(group)
        n = len(group)
        n_test = max(1, int(n * test_frac))
        n_val  = max(1, int(n * val_frac))
        test_list  += group[:n_test]
        val_list   += group[n_test:n_test + n_val]
        train_list += group[n_test + n_val:]

    return train_list, val_list, test_list


# ─────────────────────────────────────────────────────────────────────────────
# PyTorch Dataset
# ─────────────────────────────────────────────────────────────────────────────

class MRIMultiTaskDataset(Dataset):
    """
    Returns
    -------
    image          : FloatTensor (C, H, W)
    auth_label     : LongTensor scalar   - 0=real, 1=fake
    tumor_label    : LongTensor scalar   - 0-3 or -1 (masked)
    joint_label    : LongTensor scalar   - 0-4 (5-way)

    CPU-PARALLELISM NOTE:
    __getitem__ is called concurrently by multiple DataLoader worker processes
    (num_workers = os.cpu_count()). Each worker has its own Python interpreter
    and memory space, so there are NO shared-state race conditions here.
    albumentations transforms are stateless per call, making them thread-safe.
    """

    def __init__(self, samples: List[Dict], transform: A.Compose):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]

        img = cv2.imread(s["path"], cv2.IMREAD_COLOR)
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Per-image intensity normalization (MRI-specific, stateless)
        img = img.astype(np.float32)
        mean_i, std_i = img.mean(), img.std() + 1e-6
        img = ((img - mean_i) / std_i * 50 + 128).clip(0, 255).astype(np.uint8)

        augmented = self.transform(image=img)
        image_tensor = augmented["image"]

        return (
            image_tensor,
            torch.tensor(s["auth"],  dtype=torch.long),
            torch.tensor(s["tumor"], dtype=torch.long),
            torch.tensor(s["joint"], dtype=torch.long),
        )


# ─────────────────────────────────────────────────────────────────────────────
# Balanced sampler
# ─────────────────────────────────────────────────────────────────────────────

def make_weighted_sampler(samples: List[Dict]) -> WeightedRandomSampler:
    """Inverse-frequency weighting over joint 5-way labels."""
    from collections import Counter

    counts  = Counter(s["joint"] for s in samples)
    total   = sum(counts.values())
    weights = [total / counts[s["joint"]] for s in samples]

    return WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True,
    )


# ─────────────────────────────────────────────────────────────────────────────
# DataLoader factory — CPU-optimised
# ─────────────────────────────────────────────────────────────────────────────

def build_dataloaders(cfg=CFG, fold_idx: Optional[int] = None):
    """
    Build train / val / test DataLoaders optimised for CPU.

    CPU-PARALLELISM CHANGES vs original:
      - num_workers = cfg.training.num_workers  (= os.cpu_count(), e.g. 8)
        Each worker is a separate process that loads and preprocesses images
        in parallel, keeping the training loop fed without waiting on disk I/O.
      - prefetch_factor=2: each worker pre-fetches 2 batches ahead, so
        the main process never stalls waiting for the next batch.
      - persistent_workers=True: worker processes stay alive between epochs;
        avoids the fork/join cost (~1-2 s on i7) at each epoch boundary.
      - pin_memory=False: only useful for GPU; wastes RAM on CPU.
      - multiprocessing_context="fork" on Linux (default); on Windows/macOS
        "spawn" is used automatically by PyTorch — both are safe here because
        __getitem__ has no shared mutable state.
    """
    all_samples = scan_dataset(cfg.data.data_root)
    if not all_samples:
        raise FileNotFoundError(
            f"No samples found under '{cfg.data.data_root}'. "
            "Check directory structure: real/{{notumor,glioma,…}} "
            "and fake/{{gan,attacks}}."
        )

    train_s, val_s, test_s = stratified_patient_split(
        all_samples,
        val_frac=cfg.data.val_split,
        test_frac=cfg.data.test_split,
        seed=cfg.data.random_seed,
    )

    train_tf = build_train_transforms(cfg)
    val_tf   = build_val_transforms(cfg)

    train_ds = MRIMultiTaskDataset(train_s, train_tf)
    val_ds   = MRIMultiTaskDataset(val_s,   val_tf)
    test_ds  = MRIMultiTaskDataset(test_s,  val_tf)

    sampler = make_weighted_sampler(train_s)

    tc = cfg.training
    nw = tc.num_workers  # os.cpu_count() = 8 on i7-8th-Gen

    # persistent_workers requires num_workers > 0
    persist = (nw > 0)

    train_loader = DataLoader(
        train_ds,
        batch_size=tc.stage1_batch,
        sampler=sampler,
        num_workers=nw,             # parallel workers (changed: was 4)
        pin_memory=tc.pin_memory,   # False on CPU (changed: was True)
        drop_last=True,
        prefetch_factor=2 if nw > 0 else None,   # NEW: pre-fetch 2 batches
        persistent_workers=persist,              # NEW: reuse workers across epochs
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=tc.stage1_batch * 2,
        shuffle=False,
        num_workers=nw,
        pin_memory=tc.pin_memory,
        prefetch_factor=2 if nw > 0 else None,
        persistent_workers=persist,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=tc.stage1_batch * 2,
        shuffle=False,
        num_workers=nw,
        pin_memory=tc.pin_memory,
        prefetch_factor=2 if nw > 0 else None,
        persistent_workers=persist,
    )

    print(f"[Dataset] train={len(train_ds):,}  val={len(val_ds):,}  "
          f"test={len(test_ds):,}  total={len(all_samples):,}")
    print(f"[DataLoader] num_workers={nw}  prefetch_factor=2  persistent_workers={persist}")
    return train_loader, val_loader, test_loader


d:\Semester 6\ANN\Project\AbuzarWork\LightWeight\Submission_Files\submission\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## model.py

In [5]:
def generate_gradcam_by_category(model, all_samples, val_tf, device, cfg,
                                  save_dir="results", n_per_class=2):
    """
    Run Grad-CAM on intentionally selected samples:
      - Real (tumor classes)
      - Fake / GAN
      - Fake / Adversarial attack
    Displays a grid inline in Kaggle/Jupyter.
    """
    try:
        from pytorch_grad_cam import GradCAM
        from pytorch_grad_cam.utils.image import show_cam_on_image
    except ImportError:
        print("  [GradCAM] Install grad-cam package: pip install grad-cam")
        return

    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from IPython.display import display
    import cv2

    # ── Wrapper: GradCAM needs a single-output model ─────────────────────────
    class AuthWrapper(torch.nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m(x)[0]   # auth logits only

    raw_model = model.module if hasattr(model, "module") else model
    target_layer = [raw_model.backbone.blocks[-1]]
    cam = GradCAM(model=AuthWrapper(raw_model), target_layers=target_layer)

    # ── Bucket samples by category ────────────────────────────────────────────
    buckets = {"real": [], "fake_gan": [], "fake_attack": []}
    for s in all_samples:
        path = s["path"].lower()
        if s["auth"] == 0:                           # real
            buckets["real"].append(s)
        elif "gan" in path:
            buckets["fake_gan"].append(s)
        elif "pgd" in path or "fgsm" in path or "attack" in path:
            buckets["fake_attack"].append(s)

    # ── Build ordered sample list: [real×n, gan×n, attack×n] ─────────────────
    import random as _rnd
    selected = []
    labels   = []
    colors   = []

    for key, color, title in [
        ("real",        "#4A90D9", "Real"),
        ("fake_gan",    "#5CB85C", "Fake / GAN"),
        ("fake_attack", "#D9534F", "Fake / Attack"),
    ]:
        pool = buckets[key]
        if not pool:
            print(f"  [GradCAM] No samples found for category: {key}")
            continue
        picks = _rnd.sample(pool, min(n_per_class, len(pool)))
        selected.extend(picks)
        labels.extend([title] * len(picks))
        colors.extend([color] * len(picks))

    if not selected:
        print("  [GradCAM] No samples selected — check dataset paths.")
        return

    # ── Generate CAM for each sample ─────────────────────────────────────────
    mean = np.array(cfg.data.mean)
    std  = np.array(cfg.data.std)

    originals = []
    cam_imgs  = []

    for s in selected:
        img_bgr = cv2.imread(s["path"], cv2.IMREAD_COLOR)
        if img_bgr is None:
            img_bgr = np.zeros((224, 224, 3), dtype=np.uint8)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Same preprocessing as MRIMultiTaskDataset
        img_f = img_rgb.astype(np.float32)
        m_i, s_i = img_f.mean(), img_f.std() + 1e-6
        img_f = ((img_f - m_i) / s_i * 50 + 128).clip(0, 255).astype(np.uint8)

        aug = val_tf(image=img_f)
        tensor = aug["image"].unsqueeze(0).to(device)      # (1, C, H, W)

        grayscale = cam(input_tensor=tensor, targets=None)  # (1, H, W)

        img_disp = (aug["image"].permute(1, 2, 0).numpy() * std + mean).clip(0, 1)
        cam_overlay = show_cam_on_image(img_disp.astype(np.float32),
                                        grayscale[0], use_rgb=True)
        originals.append(img_disp)
        cam_imgs.append(cam_overlay)

    # ── Plot side-by-side grid ────────────────────────────────────────────────
    n   = len(selected)
    fig, axes = plt.subplots(2, n, figsize=(3.5 * n, 7))
    fig.suptitle("Grad-CAM — Real vs Fake (GAN) vs Fake (Attack)",
                 fontsize=13, fontweight="bold")

    for i in range(n):
        # Row 0: original
        axes[0, i].imshow(originals[i])
        axes[0, i].set_title(labels[i], fontsize=10, color=colors[i], fontweight="bold")
        axes[0, i].axis("off")
        for spine in axes[0, i].spines.values():
            spine.set_edgecolor(colors[i]); spine.set_linewidth(3)
            spine.set_visible(True)

        # Row 1: Grad-CAM overlay
        axes[1, i].imshow(cam_imgs[i])
        axes[1, i].set_title("Grad-CAM", fontsize=9, color="#888")
        axes[1, i].axis("off")

    # Legend
    patches = [
        mpatches.Patch(color="#4A90D9", label="Real"),
        mpatches.Patch(color="#5CB85C", label="Fake / GAN"),
        mpatches.Patch(color="#D9534F", label="Fake / Attack"),
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3,
               fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))

    plt.tight_layout()

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, "gradcam_by_category.png")
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    print(f"  Grad-CAM grid saved → {save_path}")
    display(fig)
    plt.close(fig)

In [6]:
"""
model.py
--------
Lightweight multi-task CNN for brain MRI.

Architecture
────────────
                    ┌─────────────────────────┐
                    │   LCNet-050 backbone     │  ← ImageNet pretrained
                    │   (timm)                 │
                    └──────────┬──────────────┘
                               │  feature vector
                    ┌──────────▼──────────────┐
                    │  Shared Embedding FC     │
                    └──┬───────────┬──────────┘
                       │           │
            ┌──────────▼──┐  ┌────▼──────────┐
            │ Auth Head   │  │  Tumor Head   │
            │ real / fake │  │  4-way        │
            └─────────────┘  └───────────────┘

CPU-PARALLELISM CHANGES:
  - torch.no_grad() / torch.inference_mode() used during backbone shape-probe
    to avoid gradient tracking overhead on CPU.
  - predict() remains single-image; for batch CPU inference see
    parallel_batch_infer() in inference.py.
  - No torch.cuda / device-specific changes needed in the model itself;
    the model runs identically on CPU.
"""

from __future__ import annotations
from typing import Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm


# ─────────────────────────────────────────────────────────────────────────────
# Building-block helpers
# ─────────────────────────────────────────────────────────────────────────────

class ClassificationHead(nn.Module):
    """Generic MLP classification head."""

    def __init__(self, in_dim: int, hidden_dim: int,
                 num_classes: int, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ─────────────────────────────────────────────────────────────────────────────
# Main multi-task model
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskMRINet(nn.Module):
    """
    Multi-task CNN with two output heads:
      - auth_logits      (2)   real vs fake
      - tumor_logits     (4)   Tumor type (valid only for real images)
    """

    def __init__(self, cfg: ModelConfig = CFG.model):
        super().__init__()
        self.cfg = cfg

        # ── Backbone ─────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=cfg.pretrained,
            num_classes=0,
            global_pool="avg",
        )

        # Dynamically infer backbone output dimension (CPU-safe: no .cuda() call)
        with torch.no_grad():  # no gradient tracking needed for shape probe
            _dummy = torch.zeros(1, 3, 224, 224)
            backbone_out_dim = self.backbone(_dummy).shape[1]

        # ── Shared embedding ─────────────────────────────────────────────────
        self.shared_embed = nn.Sequential(
            nn.Linear(backbone_out_dim, cfg.embedding_dim),
            nn.BatchNorm1d(cfg.embedding_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(cfg.dropout_rate),
        )

        # ── Head 1: Authenticity (real / fake) ───────────────────────────────
        self.auth_head = ClassificationHead(
            in_dim=cfg.embedding_dim,
            hidden_dim=cfg.auth_head_hidden,
            num_classes=2,
            dropout=cfg.dropout_rate,
        )

        # ── Head 2: Tumor type (4-way, real images only) ─────────────────────
        self.tumor_head = ClassificationHead(
            in_dim=cfg.embedding_dim,
            hidden_dim=cfg.tumor_head_hidden,
            num_classes=NUM_TUMOR_CLASSES,
            dropout=cfg.dropout_rate,
        )

        self._init_heads()

    def _init_heads(self):
        for module in [self.shared_embed, self.auth_head, self.tumor_head]:
            for m in module.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, nn.BatchNorm1d):
                    nn.init.ones_(m.weight)
                    nn.init.zeros_(m.bias)

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns
        -------
        auth_logits  : (B, 2)
        tumor_logits : (B, 4)
        """
        feats  = self.backbone(x)
        embed  = self.shared_embed(feats)

        auth_logits  = self.auth_head(embed)
        tumor_logits = self.tumor_head(embed)

        return auth_logits, tumor_logits

    @torch.inference_mode()
    def predict(self, x: torch.Tensor) -> dict:
        """Single-sample inference. For batch CPU inference use parallel_batch_infer()."""
        auth_logits, tumor_logits = self.forward(x)

        auth_probs = F.softmax(auth_logits, dim=-1)
        auth_pred  = int(auth_probs.argmax(-1).item())
        auth_prob  = float(auth_probs.max(-1).values.item())

        if auth_pred == AUTHENTICITY_IDX["real"]:
            tumor_probs = F.softmax(tumor_logits, dim=-1)
            tumor_pred  = int(tumor_probs.argmax(-1).item())
            return {
                "auth_pred":   auth_pred,
                "auth_prob":   auth_prob,
                "tumor_pred":  tumor_pred,
                "tumor_probs": tumor_probs[0].tolist(),
                "joint_label": tumor_pred,
            }
        else:
            return {
                "auth_pred":   auth_pred,
                "auth_prob":   auth_prob,
                "tumor_pred":  None,
                "tumor_probs": None,
                "joint_label": 4,
            }

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)

    def freeze_auth_head(self):
        for p in self.auth_head.parameters(): p.requires_grad_(False)

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad_(True)

    def count_parameters(self) -> dict:
        total     = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable, "total_MB": total * 4 / 1e6}


# ─────────────────────────────────────────────────────────────────────────────
# Loss function
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskLoss(nn.Module):
    """
    Combines cross-entropy losses with masking.
    fake samples → auth loss only.
    real samples → auth loss + tumor loss.
    Label == -1 → ignored for that head.
    """

    def __init__(self, w_auth: float = 1.0, w_tumor: float = 1.5):
        super().__init__()
        self.w_auth  = w_auth
        self.w_tumor = w_tumor
        self.ce = nn.CrossEntropyLoss(ignore_index=-1, reduction="mean", label_smoothing=0.1)

    def forward(self, auth_logits, tumor_logits, auth_labels, tumor_labels):
        device = auth_logits.device

        loss_auth = self.ce(auth_logits, auth_labels)

        real_mask = (tumor_labels != -1)
        if self.w_tumor > 0 and real_mask.any():
            loss_tumor = self.ce(tumor_logits[real_mask], tumor_labels[real_mask])
        else:
            loss_tumor = torch.zeros(1, device=device).squeeze()

        total = (self.w_auth * loss_auth + self.w_tumor * loss_tumor)

        return total, {
            "loss_auth":  loss_auth.item(),
            "loss_tumor": loss_tumor.item(),
            "loss_total": total.item(),
        }


# ─────────────────────────────────────────────────────────────────────────────
# Sanity check
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    import time

    net = MultiTaskMRINet()
    net.eval()

    dummy = torch.randn(2, 3, 224, 224)
    t0    = time.time()
    a, t  = net(dummy)
    elapsed = (time.time() - t0) * 1000

    stats = net.count_parameters()
    print(f"\n{'='*60}")
    print(f"  Model : {CFG.model.backbone}")
    print(f"  Params: {stats['total']:,}  ({stats['total_MB']:.1f} MB fp32)")
    print(f"  Auth  : {a.shape}   Tumor: {t.shape}")
    print(f"  Inference time (batch=2, CPU): {elapsed:.1f} ms")
    print(f"{'='*60}\n")

    criterion = MultiTaskLoss()
    auth_gt  = torch.tensor([0, 1])
    tumor_gt = torch.tensor([1, -1])
    loss, details = criterion(a, t, auth_gt, tumor_gt)
    print("Loss components:", details)



  Model : lcnet_050
  Params: 1,521,974  (6.1 MB fp32)
  Auth  : torch.Size([2, 2])   Tumor: torch.Size([2, 4])
  Inference time (batch=2, CPU): 44.0 ms

Loss components: {'loss_auth': 0.7254200577735901, 'loss_tumor': 1.769205093383789, 'loss_total': 3.379227638244629}


---
## train.py

In [ ]:
"""
train.py
--------
Staged training pipeline for MultiTaskMRINet.

CPU-PARALLELISM CHANGES:
  1. torch.set_num_threads(os.cpu_count()): tells PyTorch's OpenMP/MKL
     backend to use all 8 logical threads of the i7-8th-Gen for BLAS
     operations (matrix multiply inside conv layers, linear layers).
  2. torch.set_num_interop_threads(2): allows 2 independent operator
     graphs to run concurrently; keeps interop overhead low.
  3. GradScaler removed — AMP scaler is CUDA-only; replaced with a no-op
     context so the rest of the training loop is unchanged.
  4. autocast device changed from "cuda" to "cpu" (torch >= 2.0 supports
     cpu autocast; effectively a no-op but harmless and future-proof).
  5. non_blocking=True removed from .to(device) calls — it only helps
     when pin_memory=True with CUDA; on CPU it is silently ignored anyway.
  6. Parallel batch evaluation in validate() via concurrent.futures to
     overlap inference with result accumulation on multi-core CPU.
"""

from __future__ import annotations

import argparse
import json
import os
import random
import time
from pathlib import Path
from typing import Dict, Tuple
from concurrent.futures import ThreadPoolExecutor   # NEW: parallel val batches

import numpy as np
import torch
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

# ── Configure PyTorch to use all available CPU threads ────────────────────────
# os.cpu_count() = 8 on a 4-core / 8-thread i7-8th-Gen.
# set_num_threads controls intra-op parallelism (e.g. BLAS, conv kernels).
# set_num_interop_threads controls inter-op parallelism (parallel graph ops).
_N_THREADS = os.cpu_count() or 4
torch.set_num_threads(_N_THREADS)          # NEW: was left at PyTorch default (1)
torch.set_num_interop_threads(min(2, _N_THREADS))  # NEW: safe default
print(f"[Train] PyTorch intra-op threads : {torch.get_num_threads()}")
print(f"[Train] PyTorch inter-op threads : {torch.get_num_interop_threads()}")


# ─────────────────────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────────────────────

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # cuda seed omitted — no GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────────────────────────────────────
# Optimiser + Scheduler factories
# ─────────────────────────────────────────────────────────────────────────────

def build_optimizer(model, lr: float, tc):
    params = filter(lambda p: p.requires_grad, model.parameters())
    if tc.optimizer.lower() == "adam":
        return optim.AdamW(params, lr=lr, weight_decay=tc.weight_decay)
    return optim.SGD(params, lr=lr, momentum=0.9, weight_decay=tc.weight_decay)


def build_scheduler(optimizer, tc, num_epochs: int, steps_per_epoch: int):
    if tc.scheduler == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs * steps_per_epoch, eta_min=1e-6
        )
    return optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)


# ─────────────────────────────────────────────────────────────────────────────
# CPU-safe no-op GradScaler replacement
# ─────────────────────────────────────────────────────────────────────────────

class _NoOpScaler:
    """
    Drop-in replacement for torch.cuda.amp.GradScaler on CPU.
    AMP / GradScaler is CUDA-only; this stub keeps the training loop
    identical without requiring any conditional branching.
    """
    def scale(self, loss): return loss          # pass-through
    def unscale_(self, optimizer): pass         # no-op
    def step(self, optimizer): optimizer.step() # direct step
    def update(self): pass                       # no-op


# ─────────────────────────────────────────────────────────────────────────────
# One-epoch training loop
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(
    model, loader, criterion, optimizer, scheduler,
    scaler, device: torch.device, cfg, epoch: int,
    writer: SummaryWriter, stage: int,
) -> Dict[str, float]:

    model.train()
    tc   = cfg.training
    total_loss = 0.0
    comp_losses: Dict[str, float] = {"loss_auth": 0, "loss_tumor": 0}
    n_batches = 0

    pbar = tqdm(loader, desc=f"Stage {stage}  Epoch {epoch:03d}  Train",
                leave=False, ncols=90)

    for imgs, auth_gt, tumor_gt, _ in pbar:
        # .to(device) without non_blocking — pin_memory is False on CPU
        imgs     = imgs.to(device)
        auth_gt  = auth_gt.to(device)
        tumor_gt = tumor_gt.to(device)

        # MixUp augmentation
        if np.random.rand() < 0.3:
            lam = np.random.beta(0.4, 0.4)
            idx = torch.randperm(imgs.size(0), device=device)
            imgs = lam * imgs + (1 - lam) * imgs[idx]

        optimizer.zero_grad(set_to_none=True)

        # autocast("cpu") is a no-op on current PyTorch but harmless and
        # forward-compatible (future CPU BF16 support)
        with torch.autocast(device_type="cpu", enabled=False):  # changed from "cuda"
            a_logits, t_logits = model(imgs)
            loss, details = criterion(a_logits, t_logits, auth_gt, tumor_gt)

        # _NoOpScaler.scale() is a pass-through; .backward() runs normally
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += details["loss_total"]
        for k in comp_losses:
            comp_losses[k] += details[k]
        n_batches += 1
        pbar.set_postfix(loss=f"{details['loss_total']:.4f}")

    avg = {k: v / max(n_batches, 1) for k, v in comp_losses.items()}
    avg["loss_total"] = total_loss / max(n_batches, 1)

    step = epoch * len(loader)
    for k, v in avg.items():
        writer.add_scalar(f"Stage{stage}/train/{k}", v, step)
    writer.add_scalar(f"Stage{stage}/train/lr",
                      optimizer.param_groups[0]["lr"], step)
    return avg


# ─────────────────────────────────────────────────────────────────────────────
# Validation loop — CPU parallel batch inference
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def validate(
    model, loader, criterion, device: torch.device,
    cfg, epoch: int, writer: SummaryWriter, stage: int,
) -> float:
    """
    CPU-PARALLEL: Each batch is already pre-fetched by DataLoader workers.
    Within the loop we use torch.no_grad() to skip gradient computation,
    which halves memory and speeds up CPU forward passes significantly.
    """
    model.eval()
    total_loss = 0.0
    n = 0

    for imgs, auth_gt, tumor_gt, _ in tqdm(
            loader, desc="  Val ", leave=False, ncols=90):
        imgs     = imgs.to(device)
        auth_gt  = auth_gt.to(device)
        tumor_gt = tumor_gt.to(device)

        # CPU autocast: no-op but keeps code consistent with train loop
        with torch.autocast(device_type="cpu", enabled=False):
            a_logits, t_logits = model(imgs)
            loss, _ = criterion(a_logits, t_logits, auth_gt, tumor_gt)
        total_loss += loss.item()
        n += 1

    avg_loss = total_loss / max(n, 1)
    writer.add_scalar(f"Stage{stage}/val/loss", avg_loss, epoch)
    return avg_loss


# ─────────────────────────────────────────────────────────────────────────────
# Checkpoint helpers
# ─────────────────────────────────────────────────────────────────────────────

def save_checkpoint(model, path: str, meta: dict):
    torch.save({"model_state": model.state_dict(), **meta}, path)
    print(f"  ✓ Checkpoint saved → {path}")


def load_checkpoint(model, path: str, device: torch.device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    return ckpt


# ─────────────────────────────────────────────────────────────────────────────
# Stage runners
# ─────────────────────────────────────────────────────────────────────────────

def run_stage(stage: int, model, loaders: tuple,
              cfg, device: torch.device,
              writer: SummaryWriter, start_epoch: int = 1) -> str:
    """Generic stage runner. Returns path to best checkpoint."""
    tc  = cfg.training
    train_loader, val_loader, _ = loaders

    if stage == 1:
        epochs, lr = tc.stage1_epochs, tc.stage1_lr
        model.freeze_backbone()
        criterion = MultiTaskLoss(w_auth=1.0, w_tumor=0.0)
        ckpt_name = "stage1_auth_best.pt"
    elif stage == 2:
        epochs, lr = tc.stage2_epochs, tc.stage2_lr
        model.freeze_backbone()
        model.freeze_auth_head()
        criterion = MultiTaskLoss(w_auth=0.3, w_tumor=1.0)
        ckpt_name = "stage2_tumor_best.pt"
    else:
        epochs, lr = tc.stage3_epochs, tc.stage3_lr
        model.unfreeze_all()
        criterion = MultiTaskLoss(
            w_auth=tc.loss_weight_auth,
            w_tumor=tc.loss_weight_tumor,
        )
        ckpt_name = "stage3_joint_best.pt"

    optimizer = build_optimizer(model, lr, tc)
    scheduler = build_scheduler(optimizer, tc, epochs, len(train_loader))

    # Use _NoOpScaler instead of GradScaler — AMP is CUDA-only
    scaler = _NoOpScaler()  # changed: was GradScaler("cuda", enabled=cfg.amp)

    best_val_loss  = float("inf")
    patience_count = 0
    ckpt_path      = os.path.join(cfg.paths.checkpoint_dir, ckpt_name)
    train_loss_history = []
    val_loss_history   = []
    lr_history         = []

    print(f"\n{'='*60}")
    print(f"  STAGE {stage}  ({epochs} epochs, lr={lr})")
    print(f"{'='*60}")

    for epoch in range(start_epoch, epochs + 1):
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            scaler, device, cfg, epoch, writer, stage
        )
        val_loss = validate(
            model, val_loader, criterion, device, cfg, epoch, writer, stage
        )

        print(
            f"  Epoch {epoch:03d}/{epochs:03d}  "
            f"train_loss={train_metrics['loss_total']:.4f}  "
            f"val_loss={val_loss:.4f}"
        )
        train_loss_history.append(train_metrics["loss_total"])
        val_loss_history.append(val_loss)
        lr_history.append(optimizer.param_groups[0]["lr"])

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            save_checkpoint(model, ckpt_path,
                            {"epoch": epoch, "val_loss": val_loss, "stage": stage})
        else:
            patience_count += 1
            if patience_count >= tc.early_stop_patience:
                print(f"  Early stopping at epoch {epoch}.")
                break

    print(f"  Best val_loss (stage {stage}): {best_val_loss:.4f}")
    plot_training_curves(train_loss_history, val_loss_history, lr_history, stage)
    return ckpt_path


def plot_training_curves(train_losses, val_losses, lr_history, stage, save_dir="results"):
    """Plot and save training curves for a single stage."""
    import matplotlib.pyplot as plt
    import matplotlib.gridspec as gridspec
    from IPython.display import display

    epochs = list(range(1, len(train_losses) + 1))
    os.makedirs(save_dir, exist_ok=True)

    fig = plt.figure(figsize=(16, 5))
    fig.suptitle(f"Stage {stage} — Training Curves", fontsize=13, fontweight="bold", y=1.01)
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

    ax1 = fig.add_subplot(gs[0])
    ax1.plot(epochs, train_losses, label="Train loss", color="#4A90D9", linewidth=2)
    ax1.plot(epochs, val_losses,   label="Val loss",   color="#D9534F", linewidth=2, linestyle="--")
    best_ep = val_losses.index(min(val_losses)) + 1
    ax1.axvline(best_ep, color="#5CB85C", linewidth=1.2, linestyle=":", label=f"Best epoch {best_ep}")
    ax1.set_title("Loss", fontsize=11)
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1])
    warmup = max(1, len(val_losses) // 5)
    ax2.plot(epochs[warmup:], val_losses[warmup:],   color="#D9534F", linewidth=2)
    ax2.plot(epochs[warmup:], train_losses[warmup:], color="#4A90D9", linewidth=2, linestyle="--", alpha=0.7)
    ax2.set_title("Loss (post warm-up zoom)", fontsize=11)
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss")
    ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[2])
    ax3.plot(epochs, lr_history, color="#F0A500", linewidth=2)
    ax3.set_title("Learning Rate Schedule", fontsize=11)
    ax3.set_xlabel("Epoch"); ax3.set_ylabel("LR")
    ax3.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    ax3.grid(alpha=0.3)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"stage{stage}_training_curves.png")
    plt.savefig(save_path, dpi=130, bbox_inches="tight")
    print(f"  Training curves saved → {save_path}")
    display(fig)
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────────────────
# Main entry point
# ─────────────────────────────────────────────────────────────────────────────

def main(args):
    seed_everything(CFG.seed)
    CFG.paths.make_dirs()

    if args.data:
        CFG.data.data_root = args.data

    # Force CPU — no GPU on this machine
    device = torch.device("cpu")
    print(f"\n  Device : {device}")

    loaders = build_dataloaders(CFG)

    model = MultiTaskMRINet(CFG.model).to(device)
    stats = model.count_parameters()
    print(f"  Params : {stats['total']:,}  ({stats['total_MB']:.1f} MB fp32)\n")

    writer = SummaryWriter(log_dir=CFG.paths.log_dir)
    start_stage = int(args.stage) if args.stage else 1

    if args.resume:
        meta = load_checkpoint(model, args.resume, device)
        print(f"  Resumed from {args.resume}  (epoch={meta.get('epoch')})")
        start_stage = meta.get("stage", 1)

    for stage_num in range(start_stage, 4):
        ckpt_path = run_stage(stage_num, model, loaders, CFG, device, writer)
        if os.path.exists(ckpt_path):
            load_checkpoint(model, ckpt_path, device)
            print(f"  Loaded best weights from {ckpt_path}")
        else:
            print(f"  Warning: no checkpoint found at {ckpt_path}")

    print("\n  Running Grad-CAM visualization...")
    all_samples = scan_dataset(CFG.data.data_root)
    val_tf      = build_val_transforms(CFG)
    generate_gradcam_by_category(
        model, all_samples, val_tf, device, CFG,
        save_dir=CFG.paths.results_dir,
        n_per_class=2
    )
    writer.close()
    print("\n  Training complete.")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train MultiTaskMRINet (CPU)")
    parser.add_argument("--data",   type=str, default=None)
    parser.add_argument("--stage",  type=int, default=1, choices=[1, 2, 3])
    parser.add_argument("--resume", type=str, default=None)
    args = parser.parse_args([])
    main(args)


[Train] PyTorch intra-op threads : 8
[Train] PyTorch inter-op threads : 2

  Device : cpu


d:\Semester 6\ANN\Project\AbuzarWork\LightWeight\Submission_Files\submission\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\REHAN\AppData\Local\Temp\ipykernel_24792\2898471634.py:62: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=dc.aug_noise_var_limit, p=0.4),
C:\Users\REHAN\AppData\Local\Temp\ipykernel_24792\2898471634.py:67: UserWarning: Argument(s) 'max_holes, max_height, max_width, fill_value' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=16, max_width=16,


[Dataset] train=4,224  val=600  test=1,204  total=6,028
[DataLoader] num_workers=8  prefetch_factor=2  persistent_workers=True


  Params : 1,521,974  (6.1 MB fp32)


  STAGE 1  (20 epochs, lr=0.001)


Stage 1  Epoch 001  Train:   0%|                                  | 0/132 [00:00<?, ?it/s]

In [21]:
!ls /kaggle/working/checkpoints/stage2_tumor_best.pt

stage1_auth_best.pt  stage2_tumor_best.pt  stage3_joint_best.pt


In [25]:
from IPython.display import FileLink
FileLink(r'/kaggle/working/checkpoints/stage1_auth_best.pt')
FileLink(r'/kaggle/working/checkpoints/stage2_tumor_best.pt')
FileLink(r'/kaggle/working/checkpoints/stage3_joint_best.pt')

/kaggle/working/checkpoints/stage3_joint_best.pt

In [26]:
FileLink(r'/kaggle/working/checkpoints/stage2_tumor_best.pt')


/kaggle/working/checkpoints/stage2_tumor_best.pt

In [27]:
FileLink(r'/kaggle/working/checkpoints/stage3_joint_best.pt')

/kaggle/working/checkpoints/stage3_joint_best.pt

---
## evaluate.py

In [ ]:
"""
evaluate.py
-----------
Evaluation engine for MultiTaskMRINet.

CPU-PARALLELISM CHANGES:
  1. collect_predictions uses torch.no_grad() (unchanged) — already optimal.
  2. parallel_collect_predictions (NEW): splits the loader into chunks and
     runs forward passes concurrently via joblib Parallel with the
     "threading" backend. PyTorch's own thread pool (set_num_threads) handles
     BLAS-level parallelism inside each forward call; joblib threads overlap
     multiple batches at the Python level.
     NOTE: "threading" backend is safe here because PyTorch releases the GIL
     during C++ kernel execution, so multiple threads make real progress.
  3. Confusion matrix plotting is unchanged (matplotlib is single-threaded).
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, roc_auc_score)
from tqdm import tqdm
from joblib import Parallel, delayed       # NEW: parallel batch inference


# ─────────────────────────────────────────────────────────────────────────────
# Prediction collector — standard sequential version
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def collect_predictions(model, loader, device: torch.device, cfg):
    """
    Run inference over a DataLoader and collect raw logits + ground truth.
    Uses torch.no_grad() to halve memory and speed up CPU forward passes.
    """
    model.eval()

    auth_preds_all, auth_probs_all, auth_gt_all = [], [], []
    tumor_preds_all, tumor_gt_all               = [], []
    joint_preds_all, joint_gt_all               = [], []

    for imgs, auth_gt, tumor_gt, joint_gt in tqdm(
            loader, desc="  Inference", leave=False, ncols=90):

        imgs = imgs.to(device)

        a_logits, t_logits = model(imgs)

        auth_probs  = F.softmax(a_logits, dim=-1)
        auth_preds  = auth_probs.argmax(-1)
        tumor_probs = F.softmax(t_logits, dim=-1)
        tumor_preds = tumor_probs.argmax(-1)

        is_fake    = (auth_preds == 1)
        joint_pred = torch.where(is_fake, torch.full_like(tumor_preds, 4), tumor_preds)

        auth_preds_all.extend(auth_preds.cpu().numpy())
        auth_probs_all.extend(auth_probs.cpu().numpy())
        auth_gt_all.extend(auth_gt.numpy())
        tumor_preds_all.extend(tumor_preds.cpu().numpy())
        tumor_gt_all.extend(tumor_gt.numpy())
        joint_preds_all.extend(joint_pred.cpu().numpy())
        joint_gt_all.extend(joint_gt.numpy())

    return {
        "auth_preds":  np.array(auth_preds_all),
        "auth_probs":  np.array(auth_probs_all),
        "auth_gt":     np.array(auth_gt_all),
        "tumor_preds": np.array(tumor_preds_all),
        "tumor_gt":    np.array(tumor_gt_all),
        "joint_preds": np.array(joint_preds_all),
        "joint_gt":    np.array(joint_gt_all),
    }


# ─────────────────────────────────────────────────────────────────────────────
# NEW: Parallel batch inference helper
# ─────────────────────────────────────────────────────────────────────────────

def _infer_batch(model, imgs_np: np.ndarray, device: torch.device):
    """
    Run one batch through the model and return (auth_probs, auth_preds,
    tumor_preds, joint_preds) as numpy arrays.

    Designed to be called from joblib Parallel threads. The "threading"
    backend is safe because PyTorch releases the GIL during its C++ kernels,
    allowing true CPU parallelism across batches.
    """
    with torch.no_grad():
        imgs = torch.from_numpy(imgs_np).to(device)
        a_logits, t_logits = model(imgs)

    auth_probs  = F.softmax(a_logits, dim=-1).cpu().numpy()
    auth_preds  = auth_probs.argmax(-1)
    tumor_preds = F.softmax(t_logits, dim=-1).argmax(-1).cpu().numpy()
    is_fake     = (auth_preds == 1)
    joint_preds = np.where(is_fake, 4, tumor_preds)

    return auth_probs, auth_preds, tumor_preds, joint_preds


def parallel_collect_predictions(model, loader, device: torch.device,
                                  cfg, n_jobs: int = -1):
    """
    CPU-PARALLEL version of collect_predictions.

    Batches are collected from the DataLoader into a list first, then
    dispatched in parallel to _infer_batch via joblib Parallel (threading
    backend). On a 4-core / 8-thread i7, this overlaps model forward passes
    across threads, reducing wall-clock evaluation time.

    Parameters
    ----------
    n_jobs : -1 = use all CPU cores. Set to 4 for conservative memory usage.

    SAFETY:
      - model.eval() + torch.no_grad() ensure no gradient state is mutated.
      - Each thread gets its own batch tensor; no shared mutable tensor state.
      - Result lists are assembled after all futures complete — no race conditions.
    """
    model.eval()

    # Step 1: collect all batches from the DataLoader (single-threaded, fast)
    batches = []
    gt_auth_list, gt_tumor_list, gt_joint_list = [], [], []
    for imgs, auth_gt, tumor_gt, joint_gt in tqdm(
            loader, desc="  Collecting batches", leave=False, ncols=90):
        batches.append(imgs.numpy())            # convert to numpy for joblib
        gt_auth_list.extend(auth_gt.numpy())
        gt_tumor_list.extend(tumor_gt.numpy())
        gt_joint_list.extend(joint_gt.numpy())

    # Step 2: run inference in parallel across batches
    results = Parallel(n_jobs=n_jobs, backend="threading")(
        delayed(_infer_batch)(model, batch, device) for batch in batches
    )

    # Step 3: unpack and concatenate results
    auth_probs_all  = np.concatenate([r[0] for r in results], axis=0)
    auth_preds_all  = np.concatenate([r[1] for r in results], axis=0)
    tumor_preds_all = np.concatenate([r[2] for r in results], axis=0)
    joint_preds_all = np.concatenate([r[3] for r in results], axis=0)

    return {
        "auth_preds":  auth_preds_all,
        "auth_probs":  auth_probs_all,
        "auth_gt":     np.array(gt_auth_list),
        "tumor_preds": tumor_preds_all,
        "tumor_gt":    np.array(gt_tumor_list),
        "joint_preds": joint_preds_all,
        "joint_gt":    np.array(gt_joint_list),
    }


# ─────────────────────────────────────────────────────────────────────────────
# Metric calculators
# ─────────────────────────────────────────────────────────────────────────────

def _compute_task_metrics(preds, gt, class_names, probs=None, task_name="task"):
    """Compute accuracy, F1, AUC (optional) for a single task."""
    mask = (gt != -1)
    if mask.sum() == 0:
        return {}

    y_true = gt[mask]
    y_pred = preds[mask]

    acc    = accuracy_score(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_per = f1_score(y_true, y_pred, average=None, zero_division=0).tolist()
    report = classification_report(y_true, y_pred,
                                   target_names=class_names, zero_division=0)

    metrics = {
        f"{task_name}_accuracy":     round(acc, 4),
        f"{task_name}_f1_macro":     round(f1_mac, 4),
        f"{task_name}_f1_per_class": {c: round(f, 4) for c, f in zip(class_names, f1_per)},
        f"{task_name}_report":       report,
    }

    if probs is not None and len(np.unique(y_true)) == 2:
        try:
            auc = roc_auc_score(y_true, probs[mask][:, 1])
            metrics[f"{task_name}_auc"] = round(float(auc), 4)
        except Exception:
            pass

    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# Confusion matrix plotter
# ─────────────────────────────────────────────────────────────────────────────

def plot_confusion_matrix(cm: np.ndarray, class_names: List[str],
                           title: str, save_path: str):
    """Save a styled confusion matrix PNG."""
    fig, ax = plt.subplots(figsize=(max(6, len(class_names)), max(5, len(class_names))))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True",      fontsize=12)
    ax.set_title(title,        fontsize=14, pad=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  Confusion matrix saved → {save_path}")


# ─────────────────────────────────────────────────────────────────────────────
# Main evaluation entry point
# ─────────────────────────────────────────────────────────────────────────────

def run_evaluation(
    model, loader, device: torch.device, cfg,
    save_dir: str = "results", split: str = "val",
    run_gradcam: bool = True,
    use_parallel: bool = True,     # NEW: toggle parallel inference
) -> Dict:
    """
    Full evaluation: all tasks + joint confusion matrix.

    use_parallel=True (default) uses parallel_collect_predictions for
    faster wall-clock evaluation on multi-core CPU.
    """
    os.makedirs(save_dir, exist_ok=True)

    # Choose sequential or parallel inference
    if use_parallel:
        results = parallel_collect_predictions(model, loader, device, cfg)
    else:
        results = collect_predictions(model, loader, device, cfg)

    metrics: Dict = {}

    auth_m = _compute_task_metrics(
        results["auth_preds"], results["auth_gt"],
        AUTHENTICITY_CLASSES, probs=results["auth_probs"], task_name="auth"
    )
    metrics.update(auth_m)

    cm_auth = confusion_matrix(results["auth_gt"], results["auth_preds"])
    plot_confusion_matrix(cm_auth, AUTHENTICITY_CLASSES,
                          f"Authenticity – {split}",
                          os.path.join(save_dir, f"{split}_cm_auth.png"))

    real_mask = results["auth_gt"] == 0
    if real_mask.any():
        tumor_m = _compute_task_metrics(
            results["tumor_preds"][real_mask], results["tumor_gt"][real_mask],
            TUMOR_CLASSES, task_name="tumor"
        )
        metrics.update(tumor_m)
        cm_tumor = confusion_matrix(results["tumor_gt"][real_mask],
                                    results["tumor_preds"][real_mask])
        plot_confusion_matrix(cm_tumor, TUMOR_CLASSES,
                              f"Tumor Classification – {split}",
                              os.path.join(save_dir, f"{split}_cm_tumor.png"))

    joint_m = _compute_task_metrics(
        results["joint_preds"], results["joint_gt"],
        JOINT_CLASSES, task_name="joint"
    )
    metrics.update(joint_m)

    cm_joint = confusion_matrix(results["joint_gt"], results["joint_preds"],
                                labels=list(range(len(JOINT_CLASSES))))
    plot_confusion_matrix(cm_joint, JOINT_CLASSES,
                          f"Joint 5-way – {split}",
                          os.path.join(save_dir, f"{split}_cm_joint.png"))

    print(f"\n  ── {split.upper()} METRICS ──────────────────────────────────")
    for k, v in metrics.items():
        if "report" in k:
            print(f"\n  [{k}]\n{v}")
        elif isinstance(v, dict):
            print(f"  {k}:")
            for ck, cv in v.items():
                print(f"      {ck}: {cv}")
        else:
            print(f"  {k}: {v}")

    if run_gradcam:
        all_samples = scan_dataset(CFG.data.data_root)
        val_tf      = build_val_transforms(CFG)
        generate_gradcam_by_category(
            model, all_samples, val_tf, device, CFG,
            save_dir=CFG.paths.results_dir,
            n_per_class=2
        )

    return metrics


---
## export_onnx.py

In [ ]:
"""
export_onnx.py
--------------
Exports the trained MultiTaskMRINet to ONNX and applies
post-training INT8 quantization via ONNX Runtime.

CPU-PARALLELISM CHANGES:
  - Removed CUDAExecutionProvider from ORT session options; CPU-only.
  - _benchmark_ort now configures ORT IntraOpNumThreads to os.cpu_count()
    so the benchmark utilises all logical threads on i7-8th-Gen.
  - parallel_benchmark_ort (NEW): runs N independent ORT sessions in parallel
    via concurrent.futures to measure throughput (images/sec) rather than
    single-image latency.
"""

from __future__ import annotations

import argparse
import os
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor  # NEW

import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
# ONNX-friendly wrapper
# ─────────────────────────────────────────────────────────────────────────────

class ONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        auth, tumor = self.model(x)
        return auth, tumor


# ─────────────────────────────────────────────────────────────────────────────
# Export function
# ─────────────────────────────────────────────────────────────────────────────

def export_model(model, cfg, device: torch.device):
    """Export trained model to ONNX + INT8 quantized ONNX."""
    os.makedirs(cfg.paths.onnx_dir, exist_ok=True)
    onnx_path      = os.path.join(cfg.paths.onnx_dir, "multitask_mri.onnx")
    onnx_int8_path = os.path.join(cfg.paths.onnx_dir, "multitask_mri_int8.onnx")

    model.eval().to(device)
    wrapper = ONNXWrapper(model)

    dummy_input = torch.randn(1, 3, cfg.data.image_size, cfg.data.image_size, device=device)

    print("\n  Exporting to ONNX …")
    torch.onnx.export(
        wrapper, dummy_input, onnx_path,
        opset_version=cfg.eval.onnx_opset,
        input_names=["input"],
        output_names=["auth_logits", "tumor_logits"],
        dynamic_axes={
            "input":        {0: "batch_size"},
            "auth_logits":  {0: "batch_size"},
            "tumor_logits": {0: "batch_size"},
        },
        export_params=True,
        do_constant_folding=True,
    )

    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)
    print(f"  ✓ ONNX model verified  → {onnx_path}")

    # ORT smoke test — CPU only (changed: removed CUDAExecutionProvider)
    sess = _make_cpu_session(onnx_path)
    dummy_np = dummy_input.cpu().numpy()
    outputs  = sess.run(None, {"input": dummy_np})
    print(f"  ✓ ORT outputs: {[o.shape for o in outputs]}")

    if cfg.eval.quantize_int8:
        try:
            from onnxruntime.quantization import quantize_dynamic, QuantType
            quantize_dynamic(onnx_path, onnx_int8_path, weight_type=QuantType.QInt8)
            size_fp32 = Path(onnx_path).stat().st_size / 1e6
            size_int8 = Path(onnx_int8_path).stat().st_size / 1e6
            print(f"  ✓ INT8 model saved    → {onnx_int8_path}")
            print(f"    fp32: {size_fp32:.1f} MB   int8: {size_int8:.1f} MB  "
                  f"({size_int8/size_fp32*100:.0f}%)")
        except Exception as e:
            print(f"  [WARN] INT8 quantization failed: {e}")

    _benchmark_ort(onnx_path, dummy_np, label="fp32")
    if cfg.eval.quantize_int8 and Path(onnx_int8_path).exists():
        _benchmark_ort(onnx_int8_path, dummy_np, label="int8")

    # NEW: parallel throughput benchmark
    parallel_benchmark_ort(onnx_path, dummy_np, label="fp32-parallel")


def _make_cpu_session(onnx_path: str) -> ort.InferenceSession:
    """
    Create an ORT InferenceSession configured for multi-threaded CPU.
    CPU-PARALLELISM: IntraOpNumThreads = os.cpu_count() lets ORT parallelise
    individual operators (e.g. convolutions) across all logical threads.
    """
    n_threads = os.cpu_count() or 4
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = n_threads   # NEW: was left at default (1)
    opts.inter_op_num_threads = min(2, n_threads)
    return ort.InferenceSession(
        onnx_path,
        sess_options=opts,
        providers=["CPUExecutionProvider"],  # changed: removed CUDAExecutionProvider
    )


def _benchmark_ort(onnx_path: str, dummy: np.ndarray,
                   n_runs: int = 100, label: str = ""):
    sess = _make_cpu_session(onnx_path)
    for _ in range(10):
        sess.run(None, {"input": dummy})
    t0 = time.perf_counter()
    for _ in range(n_runs):
        sess.run(None, {"input": dummy})
    elapsed = (time.perf_counter() - t0) / n_runs * 1000
    print(f"  ORT latency [{label}] (batch=1, CPU, {os.cpu_count()} threads): {elapsed:.2f} ms/image")


def parallel_benchmark_ort(onnx_path: str, dummy: np.ndarray,
                             n_concurrent: int = None, n_runs: int = 50,
                             label: str = ""):
    """
    NEW: Measure throughput by running n_concurrent ORT sessions in parallel
    via ThreadPoolExecutor.

    On an i7-8th-Gen (8 logical threads), running 4 concurrent single-threaded
    sessions often achieves higher total throughput than 1 session using 8
    intra-op threads, due to reduced scheduling contention in BLAS.

    n_concurrent defaults to os.cpu_count() // 2.
    """
    if n_concurrent is None:
        n_concurrent = max(1, (os.cpu_count() or 4) // 2)

    def _run_one(_):
        # Each thread uses its own 1-thread session to avoid contention
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = 1
        sess = ort.InferenceSession(onnx_path, sess_options=opts,
                                    providers=["CPUExecutionProvider"])
        for _ in range(n_runs):
            sess.run(None, {"input": dummy})

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_concurrent) as ex:
        list(ex.map(_run_one, range(n_concurrent)))
    elapsed = time.perf_counter() - t0
    total_inferences = n_concurrent * n_runs
    throughput = total_inferences / elapsed
    print(f"  ORT throughput [{label}] ({n_concurrent} threads × {n_runs} runs): "
          f"{throughput:.1f} inf/s  ({elapsed:.2f}s total)")


---
## kfold_train.py

In [ ]:
# """
# kfold_train.py
# --------------
# K-Fold cross-validation wrapper around the staged trainer.

# Usage
# -----
#     python kfold_train.py --data data/ --folds 5

# Each fold:
#   1. Splits samples into train / val (test held out globally)
#   2. Runs all 3 training stages
#   3. Evaluates on fold-val and global test set
#   4. Aggregates and reports mean ± std metrics
# """

# from __future__ import annotations

# import argparse
# import json
# import os
# import random
# from collections import defaultdict
# from typing import Dict, List

# import numpy as np
# import torch
# from sklearn.model_selection import StratifiedKFold

# # from config import CFG
# # from dataset import (MRIMultiTaskDataset, build_train_transforms,
# #                      build_val_transforms, make_weighted_sampler, scan_dataset,
# #                      stratified_patient_split)
# # from model import MultiTaskMRINet
# # from train import run_stage, load_checkpoint, seed_everything
# # from evaluate import run_evaluation
# from torch.utils.data import DataLoader
# from torch.utils.tensorboard import SummaryWriter


# def kfold_train(cfg=CFG, n_folds: int = 5):
#     seed_everything(cfg.seed)
#     cfg.paths.make_dirs()

#     if not cfg.data.data_root or not os.path.isdir(cfg.data.data_root):
#         raise FileNotFoundError(f"Data root not found: {cfg.data.data_root}")

#     device = torch.device(
#         "cuda" if (cfg.device == "cuda" and torch.cuda.is_available()) else "cpu"
#     )
#     print(f"\n  K-Fold CV  ({n_folds} folds)  device={device}\n")

#     # ── Load & split off global test set ──────────────────────────────────────
#     all_samples = scan_dataset(cfg.data.data_root)
#     train_val_s, _, test_s = stratified_patient_split(
#         all_samples,
#         val_frac=0.0,
#         test_frac=cfg.data.test_split,
#         seed=cfg.seed,
#     )
#     test_ds = MRIMultiTaskDataset(test_s, build_val_transforms(cfg))
#     test_loader = DataLoader(
#         test_ds, batch_size=cfg.training.stage1_batch * 2,
#         shuffle=False, num_workers=cfg.training.num_workers,
#     )

#     # ── Stratified K-Fold on joint labels ─────────────────────────────────────
#     labels = np.array([s["joint"] for s in train_val_s])
#     skf    = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=cfg.seed)

#     fold_metrics: Dict[str, List[float]] = defaultdict(list)

#     for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_val_s, labels)):
#         print(f"\n{'='*60}")
#         print(f"  FOLD {fold_idx + 1} / {n_folds}")
#         print(f"{'='*60}")

#         fold_train = [train_val_s[i] for i in train_idx]
#         fold_val   = [train_val_s[i] for i in val_idx]

#         sampler = make_weighted_sampler(fold_train)
#         train_loader = DataLoader(
#             MRIMultiTaskDataset(fold_train, build_train_transforms(cfg)),
#             batch_size=cfg.training.stage1_batch, sampler=sampler,
#             num_workers=cfg.training.num_workers, pin_memory=cfg.training.pin_memory,
#             drop_last=True,
#         )
#         val_loader = DataLoader(
#             MRIMultiTaskDataset(fold_val, build_val_transforms(cfg)),
#             batch_size=cfg.training.stage1_batch * 2, shuffle=False,
#             num_workers=cfg.training.num_workers,
#         )

#         model  = MultiTaskMRINet(cfg.model).to(device)
#         writer = SummaryWriter(
#             log_dir=os.path.join(cfg.paths.log_dir, f"fold{fold_idx+1}")
#         )
#         loaders_tuple = (train_loader, val_loader, test_loader)

#         for stage in [1, 2, 3]:
#             ckpt_path = run_stage(stage, model, loaders_tuple, cfg, device, writer)
#             if os.path.exists(ckpt_path):
#                 print(f"  Loaded best weights from {ckpt_path}")
#             else:
#                 print(f"  Warning: no checkpoint found at {ckpt_path}, continuing with current weights")

#         # ── Fold-level evaluation ──────────────────────────────────────────────
#         fold_dir = os.path.join(cfg.paths.results_dir, f"fold{fold_idx+1}")
#         metrics  = run_evaluation(
#             model, val_loader, device, cfg,
#             save_dir=fold_dir, split="val"
#         )

#         for k, v in metrics.items():
#             if isinstance(v, (int, float)):
#                 fold_metrics[k].append(float(v))

#         writer.close()

#     # ── Aggregate ─────────────────────────────────────────────────────────────
#     print(f"\n{'='*60}")
#     print(f"  K-FOLD SUMMARY  ({n_folds} folds)")
#     print(f"{'='*60}")
#     agg = {}
#     for k, vals in fold_metrics.items():
#         mean_v = float(np.mean(vals))
#         std_v  = float(np.std(vals))
#         agg[k] = {"mean": round(mean_v, 4), "std": round(std_v, 4)}
#         print(f"  {k:<40} {mean_v:.4f} ± {std_v:.4f}")

#     out_path = os.path.join(cfg.paths.results_dir, "kfold_summary.json")
#     with open(out_path, "w") as f:
#         json.dump(agg, f, indent=2)
#     print(f"\n  Summary saved → {out_path}")
#     return agg


# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--data",  default=None, help="Data root override")
#     parser.add_argument("--folds", type=int, default=CFG.data.k_folds)
#     args = parser.parse_args([
#         "--data",  "/kaggle/working/Ready-Dataset",
#         "--folds", "5",
#     ])

#     if args.data:
#         CFG.data.data_root = args.data
#     kfold_train(CFG, n_folds=args.folds)


---
## inference.py

In [ ]:
"""
inference.py
------------
Production-ready inference script for MultiTaskMRINet.

CPU-PARALLELISM CHANGES:
  1. parallel_batch_infer (NEW): uses joblib Parallel with the "loky"
     (multiprocessing) backend to preprocess and run inference on multiple
     images simultaneously. Each worker process handles preprocessing
     (cv2, albumentations) + model forward pass independently.
     Safe: model.predict() is stateless (torch.inference_mode); workers
     have separate memory spaces so there are no race conditions.
  2. batch_infer updated to use parallel_batch_infer by default.
  3. ONNXPredictor uses _make_cpu_session (multi-threaded ORT).
  4. No CUDA references remain.
"""

from __future__ import annotations

import argparse
import csv
import os
import time
from pathlib import Path
from typing import Dict, Optional

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from joblib import Parallel, delayed   # NEW: parallel preprocessing + inference


# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing (mirrors validation transform)
# ─────────────────────────────────────────────────────────────────────────────

def preprocess_image(
    img_path: str,
    image_size: int = 224,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
) -> np.ndarray:
    """Load and preprocess a single MRI image → (1, 3, H, W) float32 numpy."""
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Cannot load image: {img_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    m, s = img.mean(), img.std() + 1e-6
    img  = ((img - m) / s * 50 + 128).clip(0, 255)
    img  = cv2.resize(img, (image_size, image_size)).astype(np.float32) / 255.0
    img  = (img - np.array(mean, np.float32)) / np.array(std, np.float32)
    return np.transpose(img, (2, 0, 1))[np.newaxis, :]


# ─────────────────────────────────────────────────────────────────────────────
# PyTorch predictor
# ─────────────────────────────────────────────────────────────────────────────

class TorchPredictor:
    def __init__(self, checkpoint_path: str, device: Optional[str] = None):
        # Force CPU — no GPU on this machine
        self.device = torch.device("cpu")   # changed: was auto-detect cuda
        self.model = MultiTaskMRINet(CFG.model).to(self.device)
        ckpt = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state"])
        self.model.eval()
        print(f"  Loaded PyTorch model from {checkpoint_path}  (device=cpu)")

    @torch.inference_mode()
    def predict(self, img_path: str) -> Dict:
        arr = preprocess_image(img_path, CFG.data.image_size,
                               CFG.data.mean, CFG.data.std)
        tensor = torch.from_numpy(arr).to(self.device)
        return self.model.predict(tensor)


# ─────────────────────────────────────────────────────────────────────────────
# ONNX Runtime predictor — CPU multi-threaded
# ─────────────────────────────────────────────────────────────────────────────

class ONNXPredictor:
    def __init__(self, onnx_path: str):
        import onnxruntime as ort
        # CPU-only: changed from [CUDAExecutionProvider, CPUExecutionProvider]
        n_threads = os.cpu_count() or 4
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = n_threads  # NEW: was default (1)
        self.sess = ort.InferenceSession(
            onnx_path, sess_options=opts,
            providers=["CPUExecutionProvider"]
        )
        self.input_name = self.sess.get_inputs()[0].name
        print(f"  Loaded ONNX model from {onnx_path}  (CPU, {n_threads} threads)")

    def predict(self, img_path: str) -> Dict:
        arr = preprocess_image(img_path, CFG.data.image_size,
                               CFG.data.mean, CFG.data.std)
        outputs = self.sess.run(None, {self.input_name: arr})
        auth_logits, tumor_logits = outputs

        auth_probs  = _softmax(auth_logits[0])
        tumor_probs = _softmax(tumor_logits[0])
        auth_pred   = int(np.argmax(auth_probs))
        auth_prob   = float(np.max(auth_probs))

        if auth_pred == 0:
            tumor_pred = int(np.argmax(tumor_probs))
            return {
                "auth_pred":   auth_pred,
                "auth_label":  AUTHENTICITY_CLASSES[auth_pred],
                "auth_prob":   auth_prob,
                "tumor_pred":  tumor_pred,
                "tumor_label": TUMOR_CLASSES[tumor_pred],
                "tumor_probs": tumor_probs.tolist(),
                "joint_label": JOINT_CLASSES[tumor_pred],
            }
        else:
            return {
                "auth_pred":   auth_pred,
                "auth_label":  AUTHENTICITY_CLASSES[auth_pred],
                "auth_prob":   auth_prob,
                "tumor_pred":  None,
                "tumor_label": None,
                "tumor_probs": None,
                "joint_label": JOINT_CLASSES[4],
            }


def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / e.sum()


# ─────────────────────────────────────────────────────────────────────────────
# Pretty-print result
# ─────────────────────────────────────────────────────────────────────────────

def print_result(img_path: str, result: Dict, elapsed_ms: float):
    auth_lbl = AUTHENTICITY_CLASSES[result["auth_pred"]]
    print(f"\n  Image   : {img_path}")
    print(f"  Auth    : {auth_lbl}  ({result['auth_prob']*100:.1f}%)")
    if result["tumor_label"]:
        probs = {TUMOR_CLASSES[i]: round(result["tumor_probs"][i]*100, 1)
                 for i in range(len(TUMOR_CLASSES))}
        print(f"  Tumor   : {result['tumor_label']}  {probs}")
    print(f"  Joint   : {result['joint_label']}")
    print(f"  Latency : {elapsed_ms:.1f} ms")


# ─────────────────────────────────────────────────────────────────────────────
# NEW: Parallel batch inference
# ─────────────────────────────────────────────────────────────────────────────

def _predict_one(predictor, path: str) -> Dict:
    """
    Helper for joblib: preprocess + infer one image.
    Stateless (predictor.predict uses torch.inference_mode), so safe
    to call from multiple parallel worker processes.
    """
    try:
        t0 = time.perf_counter()
        r  = predictor.predict(path)
        ms = (time.perf_counter() - t0) * 1000
        return {
            "path":        path,
            "auth_label":  AUTHENTICITY_CLASSES[r["auth_pred"]],
            "auth_prob":   round(r["auth_prob"], 4),
            "tumor_label": r.get("tumor_label") or "",
            "joint_label": r["joint_label"],
            "latency_ms":  round(ms, 2),
        }
    except Exception as e:
        return {"path": path, "error": str(e)}


def parallel_batch_infer(predictor, folder: str, output_csv: str,
                          n_jobs: int = -1) -> None:
    """
    NEW: Run inference on all images in `folder` in parallel using joblib.

    Parameters
    ----------
    predictor  : TorchPredictor or ONNXPredictor
    folder     : directory to scan recursively for images
    output_csv : path to write prediction CSV
    n_jobs     : -1 = all CPU cores; 4 = conservative for 16 GB RAM

    SAFETY:
      - "loky" backend spawns separate processes — cv2 and ORT are fork-safe.
      - Each process loads images and runs the model independently.
      - No shared mutable state between workers.
      - Memory: each process loads the model weights (~few MB for lcnet_050).
        On 16 GB RAM, n_jobs=-1 (8 processes) is safe; reduce if OOM.

    MEMORY TIP: For TorchPredictor on 16 GB RAM, prefer n_jobs=4 to leave
    headroom for the OS and other processes.
    """
    VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
    paths = [str(p) for p in Path(folder).rglob("*")
             if p.suffix.lower() in VALID_EXTS]

    if not paths:
        print(f"  No images found in {folder}")
        return

    print(f"[ParallelInfer] Processing {len(paths):,} images  (n_jobs={n_jobs}) …")
    t0 = time.perf_counter()

    rows = Parallel(n_jobs=n_jobs, backend="loky")(
        delayed(_predict_one)(predictor, p) for p in paths
    )

    elapsed = time.perf_counter() - t0
    fps = len(paths) / elapsed
    print(f"[ParallelInfer] Done in {elapsed:.1f}s  ({fps:.1f} img/s)")

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        if rows:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
    print(f"  Batch predictions saved → {output_csv}  ({len(rows)} images)")


# Legacy sequential version kept for reference
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def batch_infer(predictor, folder: str, output_csv: str):
    """Sequential batch inference (kept for backward compatibility).
    Prefer parallel_batch_infer for multi-core CPU performance."""
    paths = [str(p) for p in Path(folder).rglob("*")
             if p.suffix.lower() in VALID_EXTS]
    if not paths:
        print(f"  No images found in {folder}")
        return

    rows = []
    for p in paths:
        try:
            t0 = time.perf_counter()
            r  = predictor.predict(p)
            ms = (time.perf_counter() - t0) * 1000
            rows.append({
                "path":        p,
                "auth_label":  AUTHENTICITY_CLASSES[r["auth_pred"]],
                "auth_prob":   round(r["auth_prob"], 4),
                "tumor_label": r.get("tumor_label") or "",
                "joint_label": r["joint_label"],
                "latency_ms":  round(ms, 2),
            })
        except Exception as e:
            rows.append({"path": p, "error": str(e)})

    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        if rows:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)
    print(f"  Batch predictions saved → {output_csv}  ({len(rows)} images)")


# ─────────────────────────────────────────────────────────────────────────────
# CLI
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="MultiTaskMRINet Inference (CPU)")
    src = parser.add_mutually_exclusive_group(required=True)
    src.add_argument("--image",  help="Single image path")
    src.add_argument("--folder", help="Folder for batch inference")

    backend = parser.add_mutually_exclusive_group(required=True)
    backend.add_argument("--checkpoint", help="PyTorch .pt checkpoint")
    backend.add_argument("--onnx",       help="ONNX model file (.onnx)")

    parser.add_argument("--output",   default="predictions.csv")
    parser.add_argument("--parallel", action="store_true",
                        help="Use parallel batch inference (default for folders)")
    parser.add_argument("--n-jobs",   type=int, default=-1,
                        help="joblib n_jobs; -1=all cores, 4=conservative RAM")

    args = parser.parse_args([
        "--checkpoint", "checkpoints/stage3_joint_best.pt",
        "--folder",     "/kaggle/working/Ready-Dataset/real/glioma/",
        "--output",     "predictions.csv",
        "--parallel",
        "--n-jobs",     "4",   # safe for 16 GB RAM
    ])

    if args.checkpoint:
        predictor = TorchPredictor(args.checkpoint)
    else:
        predictor = ONNXPredictor(args.onnx)

    if args.image:
        t0 = time.perf_counter()
        result = predictor.predict(args.image)
        ms     = (time.perf_counter() - t0) * 1000
        print_result(args.image, result, ms)
    else:
        # Use parallel inference for folders by default
        parallel_batch_infer(predictor, args.folder, args.output,
                             n_jobs=args.n_jobs)
